In [12]:
# ============================================================
# V2 - Step 1: 初始化BSV动态避险实验
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 项目根目录
ROOT = Path.cwd().resolve()

# 如果Notebook从pipeline目录运行，返回上一级
if ROOT.name == "pipeline":
    ROOT = ROOT.parent

print("Project root:", ROOT)

# ETF交易池
ETFS = ["TQQQ", "UVXY", "TECL", "SPXL", "TECS", "BSV"]

# 主要参数
LOOKBACK = 60
ALPHA = 1.0
TARGET_VOL = 0.30
TRADING_DAYS = 252

# 与V1一致的回测起始日期
IS_START = "2012-08-01"

# 读取V1输出的每日仓位
weights_path = (
    ROOT / "results/signals/risk_adjusted_weights.csv"
)

v1_weights = pd.read_csv(
    weights_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# 读取原始投票仓位（后续与V2比较）
raw_path = (
    ROOT / "results/signals/ensemble_weights_history.csv"
)

raw_weights = pd.read_csv(
    raw_path,
    index_col=0,
    parse_dates=True
)[ETFS]

# 读取ETF价格
price_path = ROOT / "data/raw/etf_daily.csv"

prices = pd.read_csv(
    price_path,
    header=[0, 1],
    index_col=0,
    parse_dates=True
).sort_index()

# 使用复权收盘价计算收益率
adj_close = prices["Adj Close"][ETFS]

daily_returns = adj_close.pct_change(fill_method=None)

# 保留六只ETF收益率均存在的交易日
valid_returns = daily_returns[ETFS].dropna()

# 数据检查
assert v1_weights.index.equals(raw_weights.index)
assert np.allclose(v1_weights.sum(axis=1), 1.0)
assert v1_weights.notna().all().all()

print("\n========== 数据检查 ==========")
print("原始投票仓位：", raw_weights.shape)
print("V1风险调整仓位：", v1_weights.shape)
print("ETF历史收益率：", valid_returns.shape)

print("\n========== 回测日期 ==========")
print("开始日期：", v1_weights.index.min())
print("结束日期：", v1_weights.index.max())

print("\n========== V1平均仓位 ==========")
display(
    (v1_weights.mean() * 100)
    .round(2)
    .to_frame("Average Weight (%)")
)

print("\n========== V1最新仓位 ==========")
display(
    (v1_weights.tail(1).T * 100)
    .round(2)
)

Project root: /Users/zhangjiaxin/Desktop/fe5209_vol_rotation

========== 数据检查 ==========
原始投票仓位： (3570, 6)
V1风险调整仓位： (3570, 6)
ETF历史收益率： (3641, 6)

========== 回测日期 ==========
开始日期： 2012-08-01 00:00:00
结束日期： 2026-10-13 00:00:00

========== V1平均仓位 ==========


,Average Weight (%)
TQQQ,76.05
UVXY,7.12
TECL,5.18
SPXL,0.31
TECS,5.13
BSV,6.20



========== V1最新仓位 ==========


Date,2026-10-13
TQQQ,100.0
UVXY,0.0
TECL,0.0
SPXL,0.0
TECS,0.0
BSV,0.0


In [13]:
# ============================================================
# V2 - Step 2: 计算每日组合预估波动率
# ============================================================

portfolio_vol = pd.Series(
    index=v1_weights.index,
    dtype=float,
    name="Forecast Portfolio Volatility"
)

for date in v1_weights.index:

    # 截至当前日期，过去60个交易日的收益率
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    assert len(past_returns) == LOOKBACK, (
        f"{date}: 历史数据不足"
    )

    # 60日收益率协方差矩阵
    cov_matrix = past_returns.cov().values

    # 当天V1仓位
    w = v1_weights.loc[date, ETFS].values

    # 组合预估年化波动率
    portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# 检查结果
# ------------------------------------------------------------

assert portfolio_vol.notna().all()
assert (portfolio_vol > 0).all()

print("========== V1组合预估波动率 ==========")

display(
    portfolio_vol.describe()
    .to_frame("Annualized Volatility")
    .round(4)
)

print("\n========== 最近5个交易日 ==========")

display(
    (portfolio_vol.tail(5) * 100)
    .round(2)
    .to_frame("Forecast Vol (%)")
)

print("\n========== 超过30%目标的交易日比例 ==========")

above_target = (portfolio_vol > TARGET_VOL).mean()

print(f"{above_target:.2%}")

print("\n========== 最新预估波动率 ==========")

print(f"{portfolio_vol.iloc[-1]:.2%}")

========== V1组合预估波动率 ==========


,Annualized Volatility
count,3570.0000
mean,0.4281
std,0.2329
min,0.0091
25%,0.2932
50%,0.4068
75%,0.5305
max,2.0314



========== 最近5个交易日 ==========


,Forecast Vol (%)
Date,
2026-10-07,34.91
2026-10-08,55.26
2026-10-09,55.27
2026-10-12,54.42
2026-10-13,53.56



========== 超过30%目标的交易日比例 ==========
73.59%

========== 最新预估波动率 ==========
53.56%


In [14]:
# ============================================================
# V2 - Step 3: 动态BSV避险仓位
# ============================================================

# 1. 计算风险敞口缩放系数
risk_scale = (
    TARGET_VOL / portfolio_vol
).clip(upper=1.0)

# 检查缩放系数
assert risk_scale.notna().all()
assert ((risk_scale > 0) & (risk_scale <= 1)).all()

# 2. 复制V1仓位
v2_weights = v1_weights.copy()

# 3. 只缩放非BSV资产
RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

v2_weights[RISK_ASSETS] = (
    v1_weights[RISK_ASSETS]
    .mul(risk_scale, axis=0)
)

# 4. 将释放的资金全部配置到BSV
v2_weights["BSV"] = (
    1.0 - v2_weights[RISK_ASSETS].sum(axis=1)
)

# ------------------------------------------------------------
# 5. 数据检查
# ------------------------------------------------------------

assert v2_weights.notna().all().all()

assert (v2_weights >= -1e-10).all().all()

assert np.allclose(
    v2_weights.sum(axis=1),
    1.0
), "存在总仓位不等于100%的日期"

assert (
    v2_weights["BSV"] >= v1_weights["BSV"] - 1e-10
).all(), "出现BSV仓位异常减少"

print("========== V2平均仓位 ==========")

display(
    pd.DataFrame({
        "V1 (%)": v1_weights.mean() * 100,
        "V2 (%)": v2_weights.mean() * 100
    }).round(2)
)

# ------------------------------------------------------------
# 6. 最新仓位比较
# ------------------------------------------------------------

print("\n========== 最新交易日仓位 ==========")

latest_comparison = pd.DataFrame({
    "V1 (%)": v1_weights.iloc[-1] * 100,
    "V2 (%)": v2_weights.iloc[-1] * 100
})

display(latest_comparison.round(2))

# ------------------------------------------------------------
# 7. 避险机制触发情况
# ------------------------------------------------------------

print("\n========== BSV避险机制统计 ==========")

print(
    f"触发风险缩放的交易日比例："
    f"{(risk_scale < 1).mean():.2%}"
)

print(
    f"平均风险敞口系数："
    f"{risk_scale.mean():.2%}"
)

print(
    f"V1平均BSV仓位："
    f"{v1_weights['BSV'].mean():.2%}"
)

print(
    f"V2平均BSV仓位："
    f"{v2_weights['BSV'].mean():.2%}"
)

print(
    f"V2最高BSV仓位："
    f"{v2_weights['BSV'].max():.2%}"
)

========== V2平均仓位 ==========


,V1 (%),V2 (%)
TQQQ,76.05,54.55
UVXY,7.12,5.94
TECL,5.18,3.06
SPXL,0.31,0.28
TECS,5.13,3.46
BSV,6.20,32.72



========== 最新交易日仓位 ==========


,V1 (%),V2 (%)
TQQQ,100.0,56.01
UVXY,0.0,0.00
TECL,0.0,0.00
SPXL,0.0,0.00
TECS,0.0,0.00
BSV,0.0,43.99



========== BSV避险机制统计 ==========
触发风险缩放的交易日比例：73.59%
平均风险敞口系数：73.48%
V1平均BSV仓位：6.20%
V2平均BSV仓位：32.72%
V2最高BSV仓位：100.00%


In [15]:
# ============================================================
# V2 - Step 4: 比较加入BSV前后的组合预估波动率
# ============================================================

# 1. 计算V2每日预估波动率
v2_portfolio_vol = pd.Series(
    index=v2_weights.index,
    dtype=float,
    name="V2 Forecast Volatility"
)

for date in v2_weights.index:

    # 截至当天收盘的过去60个交易日收益率
    past_returns = (
        valid_returns
        .loc[:date, ETFS]
        .tail(LOOKBACK)
    )

    # 计算协方差矩阵
    cov_matrix = past_returns.cov().values

    # 获取V2仓位
    w = v2_weights.loc[date, ETFS].values

    # 计算组合年化预估波动率
    v2_portfolio_vol.loc[date] = np.sqrt(
        TRADING_DAYS * (w @ cov_matrix @ w)
    )


# ------------------------------------------------------------
# 2. 数据检查
# ------------------------------------------------------------

assert v2_portfolio_vol.notna().all()
assert (v2_portfolio_vol >= 0).all()


# ------------------------------------------------------------
# 3. V1 vs V2波动率比较
# ------------------------------------------------------------

vol_comparison = pd.DataFrame({
    "V1": portfolio_vol,
    "V2": v2_portfolio_vol
})

print("========== V1 vs V2预估波动率 ==========")

display(
    (vol_comparison.describe() * 100)
    .round(2)
)


# ------------------------------------------------------------
# 4. 平均波动率
# ------------------------------------------------------------

print("\n========== 平均预估年化波动率 ==========")

print(f"V1: {portfolio_vol.mean():.2%}")
print(f"V2: {v2_portfolio_vol.mean():.2%}")

print(
    f"下降幅度: "
    f"{(portfolio_vol.mean() - v2_portfolio_vol.mean()):.2%}"
)


# ------------------------------------------------------------
# 5. 超过30%目标的比例
# ------------------------------------------------------------

print("\n========== 超过30%目标的交易日比例 ==========")

print(
    f"V1: {(portfolio_vol > TARGET_VOL).mean():.2%}"
)

print(
    f"V2: {(v2_portfolio_vol > TARGET_VOL).mean():.2%}"
)


# ------------------------------------------------------------
# 6. 最近5个交易日
# ------------------------------------------------------------

print("\n========== 最近5日预估波动率 ==========")

display(
    (vol_comparison.tail(5) * 100)
    .round(2)
)


# ------------------------------------------------------------
# 7. 最新交易日
# ------------------------------------------------------------

print("\n========== 最新交易日 ==========")

print(f"V1: {portfolio_vol.iloc[-1]:.2%}")
print(f"V2: {v2_portfolio_vol.iloc[-1]:.2%}")


# ------------------------------------------------------------
# 8. 是否每天都降低了预估波动率？
# ------------------------------------------------------------

lower_vol_days = (
    v2_portfolio_vol < portfolio_vol - 1e-10
).mean()

print("\n========== 风险降低的交易日比例 ==========")

print(f"{lower_vol_days:.2%}")

========== V1 vs V2预估波动率 ==========


,V1,V2
count,357000.00,357000.00
mean,42.81,26.66
std,23.29,7.49
min,0.91,0.91
25%,29.32,29.16
50%,40.68,29.93
75%,53.05,30.05
max,203.14,31.66



========== 平均预估年化波动率 ==========
V1: 42.81%
V2: 26.66%
下降幅度: 16.16%

========== 超过30%目标的交易日比例 ==========
V1: 73.59%
V2: 33.33%

========== 最近5日预估波动率 ==========


,V1,V2
Date,,
2026-10-07,34.91,30.09
2026-10-08,55.26,30.31
2026-10-09,55.27,30.32
2026-10-12,54.42,30.31
2026-10-13,53.56,30.31



========== 最新交易日 ==========
V1: 53.56%
V2: 30.31%

========== 风险降低的交易日比例 ==========
73.59%


In [16]:
# ============================================================
# V2 - Step 5: V0 / V1 / V2 回测绩效比较
# ============================================================

import json
import ast

# ------------------------------------------------------------
# 1. 读取小组原有回测函数
# ------------------------------------------------------------

def load_function(notebook_path, function_name):

    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    for cell in notebook["cells"]:

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if (
                isinstance(node, ast.FunctionDef)
                and node.name == function_name
            ):

                module = ast.Module(
                    body=[node],
                    type_ignores=[]
                )

                exec(
                    compile(module, "<notebook>", "exec"),
                    globals()
                )

                return

    raise ValueError(f"找不到函数：{function_name}")


load_function(
    ROOT / "pipeline/09_parameter_ensemble.ipynb",
    "backtest_weighted_portfolio"
)

load_function(
    ROOT / "pipeline/00_engine.ipynb",
    "metrics"
)

print("回测函数和绩效函数加载成功")


# ------------------------------------------------------------
# 2. 使用小组原来的交易成本
# ------------------------------------------------------------

TICKS = ETFS

ONE_WAY_BPS = {
    "TQQQ": 1.0,
    "UVXY": 6.0,
    "TECL": 35.0,
    "SPXL": 16.0,
    "TECS": 15.0,
    "BSV": 1.0
}


# ------------------------------------------------------------
# 3. 构造与之前一致的复权开盘价和收盘价
# ------------------------------------------------------------

price_data = prices.sort_index()

adjustment_factor = (
    price_data["Adj Close"][ETFS]
    / price_data["Close"][ETFS]
)

open_adjusted = (
    price_data["Open"][ETFS]
    * adjustment_factor
)

O = open_adjusted.reindex(raw_weights.index)[ETFS]

C = (
    price_data["Adj Close"][ETFS]
    .reindex(raw_weights.index)
)

assert O.notna().all().all()
assert C.notna().all().all()


# ------------------------------------------------------------
# 4. 回测三种策略
# ------------------------------------------------------------

strategies = {
    "V0 Original": raw_weights,
    "V1 Inverse Vol": v1_weights,
    "V2 BSV Targeting": v2_weights
}

equity_results = {}
diagnostics = {}

for name, weights in strategies.items():

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    equity_results[name] = equity
    diagnostics[name] = diag

    print(
        f"{name}: "
        f"Final Equity = {equity.iloc[-1]:.4f}"
    )


# ------------------------------------------------------------
# 5. 检查是否复现原来的V0和V1回测
# ------------------------------------------------------------

assert np.isclose(
    equity_results["V0 Original"].iloc[-1],
    4042.970467,
    rtol=1e-5
), "V0结果与之前不一致，请检查回测设置"

assert np.isclose(
    equity_results["V1 Inverse Vol"].iloc[-1],
    2790.853514,
    rtol=1e-5
), "V1结果与之前不一致，请检查回测设置"

print("\nV0、V1历史回测一致性检查通过！")


# ------------------------------------------------------------
# 6. 设定回测区间
# ------------------------------------------------------------

periods = {
    "IS": ("2012-08-01", "2019-12-31"),
    "OOS": ("2020-01-01", None),
    "OOS ex-2020": ("2021-01-01", None)
}


# ------------------------------------------------------------
# 7. 计算绩效指标
# ------------------------------------------------------------

results = []

for name, equity in equity_results.items():

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        results.append({
            "Strategy": name,
            "Period": period,
            **performance
        })


v2_performance = pd.DataFrame(results)


# ------------------------------------------------------------
# 8. 显示三组绩效比较
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        v2_performance[
            v2_performance["Period"] == period
        ].round(4)
    )

回测函数和绩效函数加载成功
V0 Original: Final Equity = 4042.9595
V1 Inverse Vol: Final Equity = 2790.8650
V2 BSV Targeting: Final Equity = 95.4776

V0、V1历史回测一致性检查通过！

========== IS ==========


,Strategy,Period,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,V0 Original,IS,0.5164,0.4386,1.1676,1.7996,-0.6015,0.8585
3,V1 Inverse Vol,IS,0.5255,0.4081,1.2405,1.8616,-0.5278,0.9956
6,V2 BSV Targeting,IS,0.3094,0.2903,1.0764,1.5394,-0.4218,0.7336



========== OOS ==========


,Strategy,Period,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
1,V0 Original,OOS,1.1504,0.5926,1.5872,2.5635,-0.4907,2.3443
4,V1 Inverse Vol,OOS,1.0196,0.5569,1.5428,2.4004,-0.4465,2.2836
7,V2 BSV Targeting,OOS,0.4507,0.3009,1.3921,2.0421,-0.2792,1.6147



========== OOS ex-2020 ==========


,Strategy,Period,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
2,V0 Original,OOS ex-2020,0.5951,0.5416,1.1329,1.7408,-0.4907,1.2127
5,V1 Inverse Vol,OOS ex-2020,0.6253,0.5348,1.1754,1.8151,-0.4465,1.4004
8,V2 BSV Targeting,OOS ex-2020,0.3471,0.2923,1.1693,1.7144,-0.2792,1.2434


In [17]:
# ============================================================
# V2 - Step 6: 目标波动率敏感性测试
# ============================================================

TARGET_VOLS = [0.30, 0.40, 0.50, 0.60]

target_results = []
target_weights = {}
target_equities = {}

RISK_ASSETS = [etf for etf in ETFS if etf != "BSV"]

# ------------------------------------------------------------
# 1. 分别测试不同的目标波动率
# ------------------------------------------------------------

for target_vol in TARGET_VOLS:

    print(f"正在测试目标波动率：{target_vol:.0%}")

    # 根据V1预估波动率计算缩放系数
    scale = (target_vol / portfolio_vol).clip(upper=1.0)

    # 从V1仓位开始调整
    weights = v1_weights.copy()

    # 缩放非BSV资产
    weights[RISK_ASSETS] = (
        v1_weights[RISK_ASSETS]
        .mul(scale, axis=0)
    )

    # 将剩余资金全部配置到BSV
    weights["BSV"] = (
        1.0 - weights[RISK_ASSETS].sum(axis=1)
    )

    # --------------------------------------------------------
    # 2. 检查仓位
    # --------------------------------------------------------

    assert weights.notna().all().all()

    assert (weights >= -1e-10).all().all()

    assert np.allclose(
        weights.sum(axis=1),
        1.0
    )

    # 保存各目标对应的仓位
    target_weights[target_vol] = weights

    # --------------------------------------------------------
    # 3. 使用原有回测函数
    # --------------------------------------------------------

    equity, diag = backtest_weighted_portfolio(
        O,
        C,
        weights,
        start=IS_START
    )

    target_equities[target_vol] = equity

    # --------------------------------------------------------
    # 4. 记录不同区间的绩效
    # --------------------------------------------------------

    for period, (start, end) in periods.items():

        performance = metrics(
            equity,
            a=start,
            b=end
        )

        target_results.append({
            "Target Vol": target_vol,
            "Period": period,
            "Average BSV": weights["BSV"].mean(),
            "Average TQQQ": weights["TQQQ"].mean(),
            **performance
        })

    print(f"目标波动率 {target_vol:.0%} 测试完成！")


# ------------------------------------------------------------
# 5. 汇总结果
# ------------------------------------------------------------

target_comparison = pd.DataFrame(target_results)

print("\n========== 全部测试结果 ==========")

display(
    target_comparison.round(4)
)


# ------------------------------------------------------------
# 6. 分别查看IS / OOS / OOS ex-2020
# ------------------------------------------------------------

for period in periods:

    print(f"\n========== {period} ==========")

    display(
        target_comparison[
            target_comparison["Period"] == period
        ].round(4)
    )


# ------------------------------------------------------------
# 7. 检查30%是否复现之前的V2结果
# ------------------------------------------------------------

print("\n========== 30%目标一致性检查 ==========")

same_weights = np.allclose(
    target_weights[0.30][ETFS].values,
    v2_weights[ETFS].values
)

print("30%目标仓位与Step 3是否一致：", same_weights)

assert same_weights, "30%目标未能复现之前的V2仓位！"

assert np.isclose(
    target_equities[0.30].iloc[-1],
    equity_results["V2 BSV Targeting"].iloc[-1],
    rtol=1e-8
), "30%目标未能复现之前的V2净值！"

print("30%目标净值一致性检查通过！")

正在测试目标波动率：30%
目标波动率 30% 测试完成！
正在测试目标波动率：40%
目标波动率 40% 测试完成！
正在测试目标波动率：50%
目标波动率 50% 测试完成！
正在测试目标波动率：60%
目标波动率 60% 测试完成！

========== 全部测试结果 ==========


,Target Vol,Period,Average BSV,Average TQQQ,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,0.3,IS,0.3272,0.5455,0.3094,0.2903,1.0764,1.5394,-0.4218,0.7336
1,0.3,OOS,0.3272,0.5455,0.4507,0.3009,1.3921,2.0421,-0.2792,1.6147
2,0.3,OOS ex-2020,0.3272,0.5455,0.3471,0.2923,1.1693,1.7144,-0.2792,1.2434
3,0.4,IS,0.2016,0.6543,0.4004,0.3544,1.1303,1.6308,-0.4491,0.8917
4,0.4,OOS,0.2016,0.6543,0.5805,0.3811,1.3968,2.0563,-0.3208,1.8096
5,0.4,OOS ex-2020,0.2016,0.6543,0.4553,0.3726,1.1968,1.7693,-0.3208,1.4192
6,0.5,IS,0.1324,0.7126,0.4540,0.3834,1.1706,1.7108,-0.4799,0.9462
7,0.5,OOS,0.1324,0.7126,0.6836,0.4444,1.3991,2.0749,-0.3543,1.9295
8,0.5,OOS ex-2020,0.1324,0.7126,0.5214,0.4361,1.1835,1.7591,-0.3543,1.4717
9,0.6,IS,0.0994,0.7379,0.4877,0.3944,1.2066,1.7788,-0.5095,0.9572



========== IS ==========


,Target Vol,Period,Average BSV,Average TQQQ,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
0,0.3,IS,0.3272,0.5455,0.3094,0.2903,1.0764,1.5394,-0.4218,0.7336
3,0.4,IS,0.2016,0.6543,0.4004,0.3544,1.1303,1.6308,-0.4491,0.8917
6,0.5,IS,0.1324,0.7126,0.4540,0.3834,1.1706,1.7108,-0.4799,0.9462
9,0.6,IS,0.0994,0.7379,0.4877,0.3944,1.2066,1.7788,-0.5095,0.9572



========== OOS ==========


,Target Vol,Period,Average BSV,Average TQQQ,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
1,0.3,OOS,0.3272,0.5455,0.4507,0.3009,1.3921,2.0421,-0.2792,1.6147
4,0.4,OOS,0.2016,0.6543,0.5805,0.3811,1.3968,2.0563,-0.3208,1.8096
7,0.5,OOS,0.1324,0.7126,0.6836,0.4444,1.3991,2.0749,-0.3543,1.9295
10,0.6,OOS,0.0994,0.7379,0.7830,0.4856,1.4378,2.1626,-0.3739,2.0940



========== OOS ex-2020 ==========


,Target Vol,Period,Average BSV,Average TQQQ,CAGR,Vol,Sharpe,Sortino,MDD,Calmar
2,0.3,OOS ex-2020,0.3272,0.5455,0.3471,0.2923,1.1693,1.7144,-0.2792,1.2434
5,0.4,OOS ex-2020,0.2016,0.6543,0.4553,0.3726,1.1968,1.7693,-0.3208,1.4192
8,0.5,OOS ex-2020,0.1324,0.7126,0.5214,0.4361,1.1835,1.7591,-0.3543,1.4717
11,0.6,OOS ex-2020,0.0994,0.7379,0.5718,0.4737,1.1939,1.7955,-0.3739,1.5291



========== 30%目标一致性检查 ==========
30%目标仓位与Step 3是否一致： True
30%目标净值一致性检查通过！


In [19]:
# 检查各个数据集的最后有效日期

print("价格CSV最后一行：", prices.index.max())

print(
    "六只ETF最后完整价格日期：",
    prices["Adj Close"][ETFS].dropna().index.max()
)

print("原始投票最后日期：", raw_weights.index.max())

print("V1仓位最后日期：", v1_weights.index.max())

print("V2仓位最后日期：", v2_weights.index.max())

print(
    "V0净值最后日期：",
    equity_results["V0 Original"].index.max()
)

价格CSV最后一行： 2026-10-16 00:00:00
六只ETF最后完整价格日期： 2026-10-15 00:00:00
原始投票最后日期： 2026-10-13 00:00:00
V1仓位最后日期： 2026-10-13 00:00:00
V2仓位最后日期： 2026-10-13 00:00:00
V0净值最后日期： 2026-10-13 00:00:00
